## 02 — Baseline: pretrained Cellpose + local ARI

Runs pretrained Cellpose on one training FOV (DAPI z2), converts masks to polygons, assigns spots by point-in-polygon, and computes local ARI against GT-derived polygons.

### Data path
Set `MERFISH_DATA_ROOT` (recommended for Modal mounts). Otherwise we default to `/scratch/pl2820/competition`.


In [ ]:
import os
import sys

import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from cellpose import models
from shapely.geometry import Polygon
from skimage import measure

from src.io import load_fov_images
from src.assign import assign_spots_to_cells
from src.evaluate import compute_ari
from src.coords import parse_boundary_polygon

DATA_ROOT = os.environ.get('MERFISH_DATA_ROOT', '/scratch/pl2820/competition')
FOV = os.environ.get('MERFISH_FOV', 'FOV_001')
Z = int(os.environ.get('MERFISH_Z', '2'))


In [ ]:
fov_dir = f"{DATA_ROOT}/train/{FOV}"
dapi, _ = load_fov_images(fov_dir)
img = dapi[Z]

cp_model = models.Cellpose(gpu=True, model_type='cyto2')
masks, flows, styles, diams = cp_model.eval(
    img,
    diameter=None,
    channels=[0, 0],
    flow_threshold=0.4,
    cellprob_threshold=0.0,
)
print('Cells detected:', int(masks.max()))


In [ ]:
meta = pd.read_csv(f"{DATA_ROOT}/reference/fov_metadata.csv").set_index('fov')
fov_x = float(meta.loc[FOV, 'fov_x'])
fov_y = float(meta.loc[FOV, 'fov_y'])
pixel_size = float(meta.loc[FOV, 'pixel_size'])

def masks_to_polygons(masks, fov_x, fov_y, pixel_size=0.109):
    polygons = {}
    for cell_int in range(1, int(masks.max()) + 1):
        cell_mask = (masks == cell_int).astype(np.uint8)
        contours = measure.find_contours(cell_mask, 0.5)
        if not contours:
            continue
        contour = max(contours, key=len)
        xs_um = fov_x + contour[:, 1] * pixel_size
        ys_um = fov_y + contour[:, 0] * pixel_size
        poly = Polygon(zip(xs_um, ys_um))
        if poly.is_valid and (not poly.is_empty) and poly.area > 0:
            polygons[f"cellpose_{cell_int}"] = poly
    return polygons

pred_polygons = masks_to_polygons(masks, fov_x, fov_y, pixel_size=pixel_size)
print('Valid cell polygons:', len(pred_polygons))


In [ ]:
spots_train = pd.read_csv(f"{DATA_ROOT}/train/ground_truth/spots_train.csv")
fov_spots = spots_train[spots_train['fov'] == FOV].copy()
fov_spots['spot_id'] = [f"{FOV}_{i}" for i in range(len(fov_spots))]

# Predicted assignments
pred_assignments = assign_spots_to_cells(fov_spots, pred_polygons)
fov_spots['pred_cluster'] = fov_spots['spot_id'].map(pred_assignments)

# GT polygons from provided boundaries (z-plane Z)
cells = pd.read_csv(f"{DATA_ROOT}/train/ground_truth/cell_boundaries_train.csv", index_col=0)
fov_cells = cells[cells.index.str.startswith(FOV)]
gt_polygons = {}
for cell_id, row in fov_cells.iterrows():
    poly = parse_boundary_polygon(str(row.get(f"boundaryX_z{Z}", '')), str(row.get(f"boundaryY_z{Z}", '')))
    if poly is not None:
        gt_polygons[cell_id] = poly

gt_assignments = assign_spots_to_cells(fov_spots, gt_polygons)
fov_spots['gt_cluster'] = fov_spots['spot_id'].map(gt_assignments)

solution = fov_spots[['spot_id', 'fov', 'gt_cluster']].rename(columns={'gt_cluster': 'cluster_id'})
submission = fov_spots[['spot_id', 'fov', 'pred_cluster']].rename(columns={'pred_cluster': 'cluster_id'})

ari = compute_ari(solution, submission)
print(f"Local ARI on {FOV} (pretrained Cellpose): {ari:.4f}")
